# Packages

In [4]:
!pip install pandas numpy requests tqdm transformers torch plotly streamlit tqdm


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# backend ETL pipeline

In [5]:
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm
from transformers import pipeline

# --- 1. CONFIGURATION & MODEL LOAD ---
UNFIT_DATA_URL = "https://services6.arcgis.com/bdPqSfflsdgFRVVM/arcgis/rest/services/Unfit_Properties/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=json"
RENTAL_REGISTRY_URL = "https://services6.arcgis.com/bdPqSfflsdgFRVVM/arcgis/rest/services/Syracuse_Rental_Registry/FeatureServer/0/query?where=1%3D1&outFields=*&outSR=4326&f=json"

print("Loading local AI model (FLAN-T5)...")
summarizer = pipeline("text-generation", model="google/flan-t5-base")

# --- 2. DATA ACQUISITION ---
def fetch_arcgis_json_data(api_url: str) -> pd.DataFrame:
    print("Fetching live Unfit Properties data...")
    try:
        response = requests.get(api_url, timeout=15)
        response.raise_for_status() 
        json_data = response.json()
        if 'features' in json_data:
            attributes_list = [feature['attributes'] for feature in json_data['features']]
            return pd.DataFrame(attributes_list)
        return pd.DataFrame()
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

def fetch_all_arcgis_data(base_url: str) -> pd.DataFrame:
    print("Fetching Rental Registry data (Handling pagination)...")
    all_attributes = []
    offset = 0
    fetching = True
    
    while fetching:
        paginated_url = f"{base_url}&resultOffset={offset}"
        try:
            response = requests.get(paginated_url, timeout=15)
            json_data = response.json()
            if 'features' in json_data:
                batch_features = json_data['features']
                all_attributes.extend([f['attributes'] for f in batch_features])
                if json_data.get('exceededTransferLimit'):
                    offset += len(batch_features)
                else:
                    fetching = False 
            else:
                fetching = False
        except Exception as e:
            fetching = False
            
    return pd.DataFrame(all_attributes)

# --- 3. DATA ENRICHMENT ---
def clean_zip_code(zip_val) -> str:
    if pd.isna(zip_val): return "UNKNOWN"
    zip_str = str(zip_val).replace('.0', '').strip()
    return zip_str[:5] if len(zip_str) >= 5 else "UNKNOWN"

def enrich_unfit_data(df_unfit: pd.DataFrame, df_rental: pd.DataFrame) -> pd.DataFrame:
    print("Applying Absentee Landlord logic...")
    columns_to_bring_over = ['SBL', 'RRisValid']
    df_merged = pd.merge(df_unfit, df_rental[columns_to_bring_over], on='SBL', how='left')
    
    df_merged['clean_owner_zip'] = df_merged['owner_zip_code'].apply(clean_zip_code)
    df_merged['clean_property_zip'] = df_merged['zip'].apply(clean_zip_code)
    
    df_merged['is_absentee'] = np.where(
        (df_merged['clean_owner_zip'] != df_merged['clean_property_zip']) & 
        (df_merged['clean_owner_zip'] != "UNKNOWN") &
        (df_merged['clean_property_zip'] != "UNKNOWN"), 
        True, False
    )
    
    df_merged['violation_date_dt'] = pd.to_datetime(df_merged['violation_date'], unit='ms', errors='coerce')
    current_date = pd.to_datetime("today")
    df_merged['days_unsafe'] = np.where(
        df_merged['status_type_name'].str.upper() == 'OPEN',
        (current_date - df_merged['violation_date_dt']).dt.days, 0 
    )
    return df_merged

# --- 4. LOCAL AI PIPELINE ---
def tag_hazard_category(text) -> str:
    if pd.isna(text): return "Unknown"
    text = str(text).upper()
    if any(word in text for word in ['SEWAGE', 'PLUMBING', 'PIPE', 'DRAIN']): return 'Plumbing/Sewage'
    elif any(word in text for word in ['FIRE', 'SMOKE', 'DETECTOR', 'ALARM']): return 'Fire Safety'
    elif any(word in text for word in ['HEAT', 'FURNACE', 'BOILER']): return 'Heating Issue'
    elif any(word in text for word in ['WATER', 'LEAK', 'ROOF', 'MOLD', 'DAMP']): return 'Water/Mold Damage'
    elif any(word in text for word in ['RODENT', 'MICE', 'RAT', 'ROACH', 'PEST', 'INFESTATION']): return 'Pest Infestation'
    elif any(word in text for word in ['STRUCTURAL', 'STAIR', 'PORCH', 'COLLAPSE', 'ROOF']): return 'Structural Hazard'
    elif any(word in text for word in ['TRASH', 'DEBRIS', 'GARBAGE', 'CLEAN']): return 'Sanitation/Debris'
    else: return 'General Code Violation'

def apply_nlp_and_llm(df: pd.DataFrame) -> pd.DataFrame:
    print("Applying NLP Hazard Tags...")
    df['hazard_tag'] = df['corrective_action'].apply(tag_hazard_category)
    
    unique_actions = df['corrective_action'].dropna().unique()
    translation_dict = {}
    
    prompt_prefix = "Briefly summarize this housing code violation in simple terms: "
    
    for action in tqdm(unique_actions, desc="Translating with local AI"):
        prompt = prompt_prefix + str(action)
        try:
            result = summarizer(prompt, max_new_tokens=15, do_sample=False)
            raw_summary = result[0]['generated_text'].strip()
            # Clean off the parroted prompt text if the model spits it back out
            translation_dict[action] = raw_summary.replace(prompt_prefix, "").strip()
        except:
            translation_dict[action] = "Requires inspection."
            
    df['plain_english_summary'] = df['corrective_action'].map(translation_dict)
    return df

# --- 5. EXECUTION ---
print("\n--- STARTING DATA PIPELINE ---")
df_unfit_raw = fetch_arcgis_json_data(UNFIT_DATA_URL)
df_rental_raw = fetch_all_arcgis_data(RENTAL_REGISTRY_URL)

if not df_unfit_raw.empty and not df_rental_raw.empty:
    df_enriched = enrich_unfit_data(df_unfit_raw, df_rental_raw)
    df_final = apply_nlp_and_llm(df_enriched)
    
    df_final.to_csv("syracuse_safe_housing_data.csv", index=False)
    print("\n--- PIPELINE COMPLETE! Data saved to syracuse_safe_housing_data.csv ---")
    display(df_final[['address', 'is_absentee', 'hazard_tag', 'plain_english_summary']].head())

Loading local AI model (FLAN-T5)...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 282/282 [00:00<00:00, 16536.79it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohe


--- STARTING DATA PIPELINE ---
Fetching live Unfit Properties data...
Fetching Rental Registry data (Handling pagination)...
Applying Absentee Landlord logic...
Applying NLP Hazard Tags...


Translating with local AI:   0%|                                                                                                                         | 0/247 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=15) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Translating with local AI: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 247/247 [01:24<00:00,  2.92it/s]


--- PIPELINE COMPLETE! Data saved to syracuse_safe_housing_data.csv ---


,address,is_absentee,hazard_tag,plain_english_summary
0,508-10 Brighton Ave W,True,Heating Issue,Unfit due to no heat
1,508-10 Brighton Ave W,True,Fire Safety,Bring all violations caused from the fire up t...
2,113 Peters St,False,Fire Safety,Property deemed unfit due to fire damage. Obta...
3,1004 First North St,True,Fire Safety,Repair fire alarm system and maintain system
4,1004 First North St,True,Heating Issue,Unfit due to no heat


# Stream Lit File Save

In [26]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# --- Configuration ---
st.set_page_config(page_title="Syracuse Housing Monitor", layout="wide", initial_sidebar_state="expanded")

# --- Enterprise CSS Overhaul ---
st.markdown("""
<style>
.stApp { background-color: #FDFDFD; }
[data-testid="stVerticalBlockBorderWrapper"] {
    background-color: #FFFFFF !important;
    border: 1px solid #EDF2F7 !important;
    border-radius: 6px !important;
    box-shadow: 0 2px 4px rgba(0, 0, 0, 0.04) !important;
    padding: 20px !important;
    margin-bottom: 20px !important;
}
.kpi-container { text-align: center; padding: 10px 0; }
.kpi-label { font-size: 0.85rem; color: #718096; font-weight: 700; text-transform: uppercase; letter-spacing: 0.1em; }
.kpi-value { font-size: 2.4rem; font-weight: 800; color: #1A202C; margin-top: -5px; }
.kpi-trend-up { color: #E53E3E; font-size: 2.4rem; } 
h1 { font-weight: 900; color: #1A202C; text-align: center; letter-spacing: -0.02em; margin-bottom: 0;}
.subtitle { text-align: center; color: #718096; font-size: 1.1rem; margin-bottom: 30px; margin-top: 5px;}
h3 { font-size: 1.05rem !important; font-weight: 700 !important; color: #2D3748 !important; border-bottom: 1px solid #E2E8F0; padding-bottom: 8px; margin-bottom: 20px !important; text-transform: uppercase; letter-spacing: 0.05em;}
.stTabs [data-baseweb="tab-list"] { gap: 30px; }
.stTabs [data-baseweb="tab"] { font-weight: 700; color: #A0AEC0; }
.stTabs [data-baseweb="tab"][aria-selected="true"] { color: #2C7A7B; border-bottom-color: #2C7A7B; }
</style>
""", unsafe_allow_html=True)

# --- Professional Palette (Ghost-Bar Fix) ---
COLOR_TEAL = "#2C7A7B" 
COLOR_CORAL = "#E53E3E" 
# We removed the ultra-light colors so they don't vanish on a white background
SCALE_TEAL = ["#4FD1C5", "#38B2AC", "#319795", "#2C7A7B", "#285E61", "#234E52"]
SCALE_RED = ["#FC8181", "#F56565", "#E53E3E", "#C53030", "#9B2C2C", "#742A2A"]

# --- Load Data ---
@st.cache_data
def load_data():
    df = pd.read_csv("syracuse_safe_housing_data.csv")
    df = df.dropna(subset=['Latitude', 'Longitude'])
    df['plain_english_summary'] = df['plain_english_summary'].fillna("Requires physical inspection.")
    df['is_registered'] = df['RRisValid'].notna()
    if 'clean_property_zip' in df.columns:
        df['clean_property_zip'] = df['clean_property_zip'].astype(str)
    return df

try:
    df = load_data()
except:
    st.error("Data connection failed.")
    st.stop()

# --- Sidebar Logic ---
st.sidebar.title("Controls")
hazard_options = ["All Hazards"] + list(df['hazard_tag'].dropna().unique())
selected_hazard = st.sidebar.selectbox("Select Hazard Type:", hazard_options)
filtered_df = df if selected_hazard == "All Hazards" else df[df['hazard_tag'] == selected_hazard]

# --- UI START ---
st.title("Syracuse Safe Housing Monitor")
st.markdown("<div class='subtitle'>A Data-Driven Lens into Housing Safety and Landlord Accountability</div>", unsafe_allow_html=True)

# --- Metric Row ---
t_unfit = len(filtered_df)
abs_pct = (filtered_df['is_absentee'].sum() / t_unfit * 100) if t_unfit > 0 else 0
open_v = filtered_df[filtered_df['status_type_name'].str.upper() == 'OPEN']
a_days = open_v['days_unsafe'].mean() if not open_v.empty else 0
clstrs = len(filtered_df.groupby('owner_name').filter(lambda x: len(x) >= 3)['owner_name'].unique())

m1, m2, m3, m4 = st.columns(4)
with m1: st.markdown(f"<div class='kpi-container'><div class='kpi-label'>Unfit Properties</div><div class='kpi-value'>{t_unfit:,}</div></div>", unsafe_allow_html=True)
with m2: st.markdown(f"<div class='kpi-container'><div class='kpi-label'>Absentee Ownership</div><div class='kpi-value kpi-trend-up'>{abs_pct:.1f}%</div></div>", unsafe_allow_html=True)
with m3: st.markdown(f"<div class='kpi-container'><div class='kpi-label'>Avg Days Unresolved</div><div class='kpi-value'>{a_days:.0f}</div></div>", unsafe_allow_html=True)
with m4: st.markdown(f"<div class='kpi-container'><div class='kpi-label'>High-Risk Owners</div><div class='kpi-value kpi-trend-up'>{clstrs}</div></div>", unsafe_allow_html=True)

st.write("")

# --- Main Layout ---
left, right = st.columns([6, 4], gap="large")

with left:
    with st.container():
        st.markdown("### 🗺️ Geographic Hotspots")
        fig_map = px.scatter_mapbox(
            filtered_df, lat="Latitude", lon="Longitude", color="is_absentee",
            color_discrete_map={True: COLOR_CORAL, False: COLOR_TEAL},
            size="days_unsafe", size_max=12, opacity=0.75, zoom=11.5,
            mapbox_style="carto-positron", hover_name="address"
        )
        fig_map.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, showlegend=False)
        st.plotly_chart(fig_map, use_container_width=True, height=520, config={'displayModeBar': False})

    st.write("")
    
    with st.container():
        st.markdown("### 🔍 Investigation Hub")
        t1, t2 = st.tabs(["Stalled Violation Leaderboard", "Owner Portfolio Lookup"])
        with t1:
            st.dataframe(open_v.sort_values('days_unsafe', ascending=False).head(15)[['address', 'owner_name', 'days_unsafe', 'plain_english_summary']], use_container_width=True, hide_index=True)
        with t2:
            owners = sorted(filtered_df['owner_name'].dropna().unique())
            owner = st.selectbox("Select Owner from view:", [""] + owners)
            if owner:
                o_df = filtered_df[filtered_df['owner_name'] == owner]
                st.info(f"**{owner}** has **{len(o_df)}** filtered violations.")
                st.dataframe(o_df[['address', 'hazard_tag', 'days_unsafe', 'plain_english_summary']], use_container_width=True, hide_index=True)

with right:
    with st.container():
        st.markdown("### 📊 Violation Categories")
        h_counts = filtered_df['hazard_tag'].value_counts().reset_index()
        fig_h = px.bar(h_counts, x='count', y='hazard_tag', orientation='h', color='count', color_continuous_scale=SCALE_TEAL)
        fig_h.update_layout(xaxis_visible=False, yaxis_title=None, margin={"r":10,"t":10,"l":10,"b":10}, template="plotly_white", coloraxis_showscale=False)
        fig_h.update_traces(texttemplate='%{x}', textposition='outside', textfont_size=12)
        st.plotly_chart(fig_h, use_container_width=True, height=280, config={'displayModeBar': False})

    with st.container():
        st.markdown("### 📝 Registry Status")
        reg = filtered_df['is_registered'].value_counts().reset_index()
        reg['label'] = reg['is_registered'].map({True: 'Registered', False: 'Unregistered'})
        
        # PLOTLY BUG FIX: Added `color='label'` so it forces the color map to work!
        fig_p = px.pie(reg, values='count', names='label', color='label', hole=0.65, 
                       color_discrete_map={'Registered': COLOR_TEAL, 'Unregistered': COLOR_CORAL})
        
        fig_p.update_layout(margin={"r":0,"t":0,"l":0,"b":0}, showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5))
        fig_p.update_traces(textinfo='percent', textfont_size=14, textposition='inside')
        st.plotly_chart(fig_p, use_container_width=True, height=240, config={'displayModeBar': False})

    with st.container():
        st.markdown("### 🏙️ Zip Code Impact")
        zips = filtered_df['clean_property_zip'].value_counts().reset_index().head(5)
        fig_z = px.bar(zips, x='clean_property_zip', y='count', color='count', color_continuous_scale=SCALE_RED)
        fig_z.update_layout(xaxis_title=None, yaxis_visible=False, margin={"r":10,"t":10,"l":10,"b":10}, template="plotly_white", coloraxis_showscale=False)
        fig_z.update_traces(texttemplate='%{y}', textposition='outside', textfont_size=12)
        st.plotly_chart(fig_z, use_container_width=True, height=200, config={'displayModeBar': False})

Overwriting app.py
